# Phase 8 — Streamlit Dashboard

- **8A.** Inspect API and define dashboard contract
- **8B.** Configuration, API client, and utilities
- **8C.** Main professional forecast page
- **8D.** Alerts and system information
- **8E.** Essential tests
- **8F.** Docker Compose and validation

## **8A.** Inspect the FastAPI service and define the dashboard contract

This inspection will confirm:

- the FastAPI base URL
- all available dashboard-facing endpoints
- OpenAPI response contracts
- forecast and summary field names
- nullable rolling-AQI behavior
- alert response structure
- timestamp formats
- freshness and readiness fields
- structured API errors
- dashboard page structure
- component inventory
- endpoint-to-component usage

No Streamlit UI is implemented in this subphase.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from IPython.display import display

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Reports directory:", REPORTS_DIR)

Project root: /home/riyan/Riyan/projects/pearls-aqi-predictor
Reports directory: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports


In [3]:
FASTAPI_ROOT_URL = "http://localhost:8000"
FASTAPI_BASE_URL = (
    f"{FASTAPI_ROOT_URL}/api/v1"
)

REQUEST_TIMEOUT_SECONDS = 10

print("FastAPI root URL:", FASTAPI_ROOT_URL)
print("FastAPI base URL:", FASTAPI_BASE_URL)

FastAPI root URL: http://localhost:8000
FastAPI base URL: http://localhost:8000/api/v1


### Confirm that FastAPI is running

Liveness confirms that the API process is available independently of forecast
freshness or artifact readiness.

In [4]:
def request_json(
    url: str,
    *,
    params: dict[str, Any] | None = None,
) -> tuple[int, dict[str, Any]]:
    """Send a GET request and return status plus JSON."""

    try:
        response = requests.get(
            url,
            params=params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Could not connect to FastAPI at {url}"
        ) from exc

    try:
        payload = response.json()
    except requests.JSONDecodeError as exc:
        raise ValueError(
            f"FastAPI returned invalid JSON from {url}"
        ) from exc

    if not isinstance(payload, dict):
        raise TypeError(
            "Expected a JSON object from "
            f"{url}, received {type(payload).__name__}."
        )

    return response.status_code, payload

In [5]:
liveness_status_code, liveness_payload = (
    request_json(
        f"{FASTAPI_BASE_URL}/health/live"
    )
)

display(
    pd.Series(
        liveness_payload,
        name="value",
    ).to_frame()
)

assert liveness_status_code == 200
assert liveness_payload["status"] == "ALIVE"

print("FastAPI liveness validation passed.")

,value
status,ALIVE
service,Pearls AQI Predictor API
version,1.0.0
timestamp_utc,2026-07-28T18:26:09.963685Z


FastAPI liveness validation passed.


### Inspect the OpenAPI contract

OpenAPI provides the authoritative list of routes, query parameters, response
schemas, enums, and nullable fields exposed by FastAPI.

The dashboard implementation should follow this contract rather than assuming
field names.

In [6]:
openapi_status_code, openapi_payload = (
    request_json(
        f"{FASTAPI_ROOT_URL}/openapi.json"
    )
)

assert openapi_status_code == 200

openapi_info = openapi_payload.get(
    "info",
    {},
)

openapi_summary = {
    "title": openapi_info.get("title"),
    "version": openapi_info.get("version"),
    "documented_path_count": len(
        openapi_payload.get("paths", {})
    ),
    "schema_count": len(
        openapi_payload
        .get("components", {})
        .get("schemas", {})
    ),
}

display(
    pd.Series(
        openapi_summary,
        name="value",
    ).to_frame()
)

,value
title,Pearls AQI Predictor API
version,1.0.0
documented_path_count,9
schema_count,22


### Extract dashboard-facing endpoints

Only versioned GET endpoints are relevant to Streamlit.

Documentation endpoints are useful for developers but are not called during
normal dashboard rendering.

In [7]:
endpoint_records = []

for path, operations in (
    openapi_payload
    .get("paths", {})
    .items()
):
    for method, operation in operations.items():
        if method.lower() != "get":
            continue

        endpoint_records.append(
            {
                "method": method.upper(),
                "path": path,
                "summary": operation.get(
                    "summary"
                ),
                "operation_id": operation.get(
                    "operationId"
                ),
                "tags": operation.get(
                    "tags",
                    [],
                ),
            }
        )

api_endpoint_inventory_df = (
    pd.DataFrame(endpoint_records)
    .sort_values("path")
    .reset_index(drop=True)
)

display(api_endpoint_inventory_df)

,method,path,summary,operation_id,tags
0,GET,/api/v1/alerts,Get forecast alert episodes,get_alert_episodes_api_v1_alerts_get,[Alerts]
1,GET,/api/v1/alerts/active,Get current and upcoming alert episodes,get_active_alerts_api_v1_alerts_active_get,[Alerts]
2,GET,/api/v1/forecast,Get the complete 72-hour forecast,get_complete_forecast_api_v1_forecast_get,[Forecast]
3,GET,/api/v1/forecast/hourly,Get filterable hourly forecasts,get_hourly_forecast_api_v1_forecast_hourly_get,[Forecast]
4,GET,/api/v1/forecast/summary,Get the forecast summary,get_forecast_summary_api_v1_forecast_summary_get,[Forecast]
5,GET,/api/v1/health/live,Check service liveness,get_liveness_api_v1_health_live_get,[Health]
6,GET,/api/v1/health/ready,Check forecast readiness,get_readiness_api_v1_health_ready_get,[Health]
7,GET,/api/v1/metadata,Get public project metadata,get_metadata_api_v1_metadata_get,[Metadata and operations]
8,GET,/api/v1/pipeline/status,Get the latest pipeline status,get_pipeline_status_api_v1_pipeline_status_get,[Metadata and operations]


In [8]:
expected_dashboard_paths = {
    "/api/v1/health/live",
    "/api/v1/health/ready",
    "/api/v1/forecast",
    "/api/v1/forecast/hourly",
    "/api/v1/forecast/summary",
    "/api/v1/alerts",
    "/api/v1/alerts/active",
    "/api/v1/metadata",
    "/api/v1/pipeline/status",
}

actual_dashboard_paths = set(
    api_endpoint_inventory_df["path"]
)

missing_dashboard_paths = sorted(
    expected_dashboard_paths.difference(
        actual_dashboard_paths
    )
)

print(
    "Missing expected dashboard paths:",
    missing_dashboard_paths,
)

assert not missing_dashboard_paths

Missing expected dashboard paths: []


### Inspect readiness and freshness

Readiness tells the dashboard whether forecast data is valid, available, and
fresh enough to present.

The dashboard should show:

- service readiness
- forecast availability
- artifact validity
- freshness status
- forecast age
- pipeline run ID
- documented limitations

A stale or unavailable forecast should produce a visible warning rather than a
broken page.

In [9]:
readiness_status_code, readiness_payload = (
    request_json(
        f"{FASTAPI_BASE_URL}/health/ready"
    )
)

print(
    "Readiness HTTP status:",
    readiness_status_code,
)

display(
    pd.json_normalize(
        readiness_payload
    ).T.rename(
        columns={0: "value"}
    )
)

Readiness HTTP status: 200


,value
status,READY_WITH_LIMITATIONS
service,Pearls AQI Predictor API
version,1.0.0
timestamp_utc,2026-07-28T18:26:10.208056Z
forecast_available,True
artifacts_valid,True
pipeline_run_id,20260728T150612Z_aqi_d93945f7
forecast_rows,72
limitations,[Indicative hourly AQI is not an official regu...
message,The latest forecast is ready with documented l...


In [10]:
readiness_contract = {
    "status": readiness_payload.get(
        "status"
    ),
    "forecast_available": (
        readiness_payload.get(
            "forecast_available"
        )
    ),
    "artifacts_valid": (
        readiness_payload.get(
            "artifacts_valid"
        )
    ),
    "pipeline_run_id": (
        readiness_payload.get(
            "pipeline_run_id"
        )
    ),
    "forecast_rows": (
        readiness_payload.get(
            "forecast_rows"
        )
    ),
    "freshness_status": (
        readiness_payload
        .get("freshness", {})
        .get("status")
    ),
    "age_hours": (
        readiness_payload
        .get("freshness", {})
        .get("age_hours")
    ),
    "message": readiness_payload.get(
        "message"
    ),
}

display(
    pd.Series(
        readiness_contract,
        name="value",
    ).to_frame()
)

,value
status,READY_WITH_LIMITATIONS
forecast_available,True
artifacts_valid,True
pipeline_run_id,20260728T150612Z_aqi_d93945f7
forecast_rows,72
freshness_status,FRESH
age_hours,3.333
message,The latest forecast is ready with documented l...


### Inspect the complete dashboard forecast response

The main forecast endpoint contains:

- project and location information
- pipeline identity
- generation and forecast timestamps
- freshness
- summary statistics
- active-alert count
- all 72 hourly forecast records
- limitations
- disclaimer

This will be the main data source for the Forecast page.

In [11]:
forecast_status_code, forecast_payload = (
    request_json(
        f"{FASTAPI_BASE_URL}/forecast"
    )
)

assert forecast_status_code == 200

forecast_top_level_schema_df = pd.DataFrame(
    [
        {
            "field": key,
            "python_type": type(
                value
            ).__name__,
        }
        for key, value
        in forecast_payload.items()
    ]
)

display(forecast_top_level_schema_df)

,field,python_type
0,project_name,str
1,description,str
2,location,dict
3,pipeline_run_id,str
4,source_phase_5_run_id,str
5,generated_at_utc,str
6,reference_time_utc,str
7,forecast_start_utc,str
8,forecast_end_utc,str
9,freshness,dict


In [12]:
hourly_forecast_records = (
    forecast_payload.get(
        "hourly_forecast",
        [],
    )
)

forecast_response_summary = {
    "pipeline_run_id": (
        forecast_payload.get(
            "pipeline_run_id"
        )
    ),
    "source_phase_5_run_id": (
        forecast_payload.get(
            "source_phase_5_run_id"
        )
    ),
    "generated_at_utc": (
        forecast_payload.get(
            "generated_at_utc"
        )
    ),
    "reference_time_utc": (
        forecast_payload.get(
            "reference_time_utc"
        )
    ),
    "forecast_start_utc": (
        forecast_payload.get(
            "forecast_start_utc"
        )
    ),
    "forecast_end_utc": (
        forecast_payload.get(
            "forecast_end_utc"
        )
    ),
    "hourly_record_count": len(
        hourly_forecast_records
    ),
    "active_alert_count": (
        forecast_payload.get(
            "active_alert_count"
        )
    ),
    "freshness_status": (
        forecast_payload
        .get("freshness", {})
        .get("status")
    ),
    "disclaimer_present": bool(
        forecast_payload.get(
            "disclaimer"
        )
    ),
}

display(
    pd.Series(
        forecast_response_summary,
        name="value",
    ).to_frame()
)

assert len(hourly_forecast_records) == 72

,value
pipeline_run_id,20260728T150612Z_aqi_d93945f7
source_phase_5_run_id,20260728T150416Z_d93945f7
generated_at_utc,2026-07-28T15:06:12.277712Z
reference_time_utc,2026-07-28T13:00:00Z
forecast_start_utc,2026-07-28T14:00:00Z
forecast_end_utc,2026-07-31T13:00:00Z
hourly_record_count,72
active_alert_count,0
freshness_status,FRESH
disclaimer_present,True


### Inspect one hourly forecast record

The dashboard charts, table, selected-hour detail, and health panel will all use
this public hourly schema.

Rolling fields must remain nullable. Missing rolling values will later display
as “Not available” rather than zero.

In [13]:
first_hourly_record = (
    hourly_forecast_records[0]
)

hourly_record_schema_df = pd.DataFrame(
    [
        {
            "field": key,
            "python_type": type(
                value
            ).__name__,
            "nullable_in_sample": (
                value is None
            ),
            "sample_value": value,
        }
        for key, value
        in first_hourly_record.items()
    ]
)

display(hourly_record_schema_df)

,field,python_type,nullable_in_sample,sample_value
0,target_time_utc,str,False,2026-07-28T14:00:00Z
1,forecast_horizon_hours,int,False,1
2,predicted_pm25_ug_m3,float,False,11.2
3,indicative_hourly_pm25_aqi,int,False,55
4,indicative_hourly_aqi_category,str,False,Moderate
5,indicative_hourly_aqi_color_hex,str,False,#FFFF00
6,rolling_24h_pm25_ug_m3,float,False,8.6375
7,rolling_24h_pm25_aqi,int,False,48
8,rolling_24h_aqi_category,str,False,Good
9,rolling_24h_aqi_color_hex,str,False,#00E400


In [14]:
required_hourly_dashboard_fields = {
    "target_time_utc",
    "forecast_horizon_hours",
    "predicted_pm25_ug_m3",
    "indicative_hourly_pm25_aqi",
    "indicative_hourly_aqi_category",
    "indicative_hourly_aqi_color_hex",
    "rolling_24h_pm25_ug_m3",
    "rolling_24h_pm25_aqi",
    "rolling_24h_aqi_category",
    "rolling_24h_aqi_color_hex",
    "rolling_24h_pm25_is_complete",
    "alert_level",
    "alert_basis",
    "alert_trigger_aqi",
    "alert_trigger_category",
    "alert_is_active",
    "sensitive_groups_alert",
    "general_population_alert",
    "hazardous_alert",
    "health_message",
    "recommended_action",
}

missing_hourly_dashboard_fields = sorted(
    required_hourly_dashboard_fields
    .difference(first_hourly_record)
)

print(
    "Missing hourly dashboard fields:",
    missing_hourly_dashboard_fields,
)

assert not missing_hourly_dashboard_fields

Missing hourly dashboard fields: []


### Validate timestamps, horizons, and nullable fields

This inspection confirms that:

- all 72 records are present
- horizons are ordered from 1 to 72
- target timestamps parse as timezone-aware UTC
- rolling values remain nullable
- chart numeric fields can be converted safely

In [15]:
hourly_forecast_df = pd.DataFrame(
    hourly_forecast_records
)

hourly_forecast_df[
    "target_time_utc"
] = pd.to_datetime(
    hourly_forecast_df[
        "target_time_utc"
    ],
    utc=True,
    errors="coerce",
)

numeric_dashboard_fields = [
    "forecast_horizon_hours",
    "predicted_pm25_ug_m3",
    "indicative_hourly_pm25_aqi",
    "rolling_24h_pm25_ug_m3",
    "rolling_24h_pm25_aqi",
    "alert_trigger_aqi",
]

for column in numeric_dashboard_fields:
    hourly_forecast_df[column] = (
        pd.to_numeric(
            hourly_forecast_df[column],
            errors="coerce",
        )
    )

display(hourly_forecast_df.head())

,target_time_utc,forecast_horizon_hours,predicted_pm25_ug_m3,indicative_hourly_pm25_aqi,indicative_hourly_aqi_category,indicative_hourly_aqi_color_hex,rolling_24h_pm25_ug_m3,rolling_24h_pm25_aqi,rolling_24h_aqi_category,rolling_24h_aqi_color_hex,...,alert_level,alert_basis,alert_trigger_aqi,alert_trigger_category,alert_is_active,sensitive_groups_alert,general_population_alert,hazardous_alert,health_message,recommended_action
0,2026-07-28 14:00:00+00:00,1,11.2,55,Moderate,#FFFF00,8.637500,48,Good,#00E400,...,NORMAL,rolling_24h_pm25_aqi,48,Good,False,False,False,False,Air quality is satisfactory and poses little o...,Normal outdoor activities may continue.
1,2026-07-28 15:00:00+00:00,2,11.2,55,Moderate,#FFFF00,8.716667,48,Good,#00E400,...,NORMAL,rolling_24h_pm25_aqi,48,Good,False,False,False,False,Air quality is satisfactory and poses little o...,Normal outdoor activities may continue.
2,2026-07-28 16:00:00+00:00,3,11.2,55,Moderate,#FFFF00,8.854167,49,Good,#00E400,...,NORMAL,rolling_24h_pm25_aqi,49,Good,False,False,False,False,Air quality is satisfactory and poses little o...,Normal outdoor activities may continue.
3,2026-07-28 17:00:00+00:00,4,11.2,55,Moderate,#FFFF00,8.954167,49,Good,#00E400,...,NORMAL,rolling_24h_pm25_aqi,49,Good,False,False,False,False,Air quality is satisfactory and poses little o...,Normal outdoor activities may continue.
4,2026-07-28 18:00:00+00:00,5,11.2,55,Moderate,#FFFF00,8.941667,49,Good,#00E400,...,NORMAL,rolling_24h_pm25_aqi,49,Good,False,False,False,False,Air quality is satisfactory and poses little o...,Normal outdoor activities may continue.


In [16]:
hourly_data_validation = {
    "rows": len(hourly_forecast_df),
    "first_horizon": int(
        hourly_forecast_df[
            "forecast_horizon_hours"
        ].min()
    ),
    "last_horizon": int(
        hourly_forecast_df[
            "forecast_horizon_hours"
        ].max()
    ),
    "unique_horizons": int(
        hourly_forecast_df[
            "forecast_horizon_hours"
        ].nunique()
    ),
    "unique_target_times": int(
        hourly_forecast_df[
            "target_time_utc"
        ].nunique()
    ),
    "missing_target_times": int(
        hourly_forecast_df[
            "target_time_utc"
        ].isna().sum()
    ),
    "missing_pm25": int(
        hourly_forecast_df[
            "predicted_pm25_ug_m3"
        ].isna().sum()
    ),
    "missing_indicative_aqi": int(
        hourly_forecast_df[
            "indicative_hourly_pm25_aqi"
        ].isna().sum()
    ),
    "incomplete_rolling_windows": int(
        (
            ~hourly_forecast_df[
                "rolling_24h_pm25_is_complete"
            ].astype(bool)
        ).sum()
    ),
    "missing_rolling_aqi": int(
        hourly_forecast_df[
            "rolling_24h_pm25_aqi"
        ].isna().sum()
    ),
}

display(
    pd.Series(
        hourly_data_validation,
        name="value",
    ).to_frame()
)

assert hourly_data_validation["rows"] == 72
assert hourly_data_validation["first_horizon"] == 1
assert hourly_data_validation["last_horizon"] == 72

,value
rows,72
first_horizon,1
last_horizon,72
unique_horizons,72
unique_target_times,72
missing_target_times,0
missing_pm25,0
missing_indicative_aqi,0
incomplete_rolling_windows,0
missing_rolling_aqi,0


### Inspect forecast summary

The summary endpoint will supply the top metric cards and compact overview
values.

The dashboard should use this saved API definition instead of independently
recalculating competing summary definitions.

In [17]:
summary_status_code, summary_payload = (
    request_json(
        f"{FASTAPI_BASE_URL}/forecast/summary"
    )
)

assert summary_status_code == 200

summary_schema_df = pd.DataFrame(
    [
        {
            "field": key,
            "python_type": type(
                value
            ).__name__,
            "value": value,
        }
        for key, value
        in summary_payload.items()
    ]
)

display(summary_schema_df)

,field,python_type,value
0,phase_6_run_id,str,20260728T150612Z_aqi_d93945f7
1,source_phase_5_run_id,str,20260728T150416Z_d93945f7
2,generated_at_utc,str,2026-07-28T15:06:12.277712Z
3,reference_time_utc,str,2026-07-28T13:00:00Z
4,forecast_start_utc,str,2026-07-28T14:00:00Z
5,forecast_end_utc,str,2026-07-31T13:00:00Z
6,forecast_rows,int,72
7,minimum_predicted_pm25_ug_m3,float,9.402523
8,maximum_predicted_pm25_ug_m3,float,15.603418
9,average_predicted_pm25_ug_m3,float,11.521162


### Confirm server-side forecast filters

FastAPI already supports horizon, category, alert-level, and alerts-only
filters.

Streamlit may still filter the loaded 72-row response locally for smooth UI
interaction, but it should not invent unsupported values.

In [18]:
hourly_endpoint_operation = (
    openapi_payload["paths"][
        "/api/v1/forecast/hourly"
    ]["get"]
)

hourly_query_parameters = []

for parameter in hourly_endpoint_operation.get(
    "parameters",
    [],
):
    schema = parameter.get(
        "schema",
        {},
    )

    hourly_query_parameters.append(
        {
            "name": parameter.get("name"),
            "required": parameter.get(
                "required",
                False,
            ),
            "type": schema.get("type"),
            "minimum": schema.get(
                "minimum"
            ),
            "maximum": schema.get(
                "maximum"
            ),
            "default": schema.get(
                "default"
            ),
        }
    )

hourly_filter_contract_df = pd.DataFrame(
    hourly_query_parameters
)

display(hourly_filter_contract_df)

,name,required,type,minimum,maximum,default
0,minimum_horizon,False,NaN,None,None,None
1,maximum_horizon,False,NaN,None,None,None
2,category,False,NaN,None,None,None
3,alert_level,False,NaN,None,None,None
4,alerts_only,False,boolean,None,None,False


### Inspect alert responses

The Alerts page will distinguish:

- all forecast alert episodes
- currently active alert episodes
- upcoming alert episodes
- no-alert state

An empty episode list is a valid response and must not be treated as an error.

In [19]:
alerts_status_code, alerts_payload = (
    request_json(
        f"{FASTAPI_BASE_URL}/alerts"
    )
)

active_alerts_status_code, active_alerts_payload = (
    request_json(
        f"{FASTAPI_BASE_URL}/alerts/active"
    )
)

assert alerts_status_code == 200
assert active_alerts_status_code == 200

alert_response_summary = {
    "all_episode_count": (
        alerts_payload.get(
            "episode_count"
        )
    ),
    "all_episodes_type": type(
        alerts_payload.get(
            "episodes"
        )
    ).__name__,
    "currently_active_count": (
        active_alerts_payload.get(
            "current_count"
        )
    ),
    "upcoming_count": (
        active_alerts_payload.get(
            "upcoming_count"
        )
    ),
    "active_episodes_type": type(
        active_alerts_payload.get(
            "episodes"
        )
    ).__name__,
}

display(
    pd.Series(
        alert_response_summary,
        name="value",
    ).to_frame()
)

,value
all_episode_count,0
all_episodes_type,list
currently_active_count,0
upcoming_count,0
active_episodes_type,list


In [20]:
alert_episode_fields = []

if alerts_payload.get("episodes"):
    alert_episode_fields = sorted(
        alerts_payload[
            "episodes"
        ][0].keys()
    )

alert_episode_contract = {
    "episodes_available": bool(
        alerts_payload.get(
            "episodes"
        )
    ),
    "episode_fields": (
        alert_episode_fields
    ),
    "empty_list_supported": isinstance(
        alerts_payload.get(
            "episodes"
        ),
        list,
    ),
}

display(
    pd.Series(
        alert_episode_contract,
        name="value",
    ).to_frame()
)

,value
episodes_available,False
episode_fields,[]
empty_list_supported,True


### Inspect public metadata

The System Status and Location sections will use this endpoint for:

- project description
- reference location
- coordinates
- pollution and weather sources
- AQI standard
- forecast scope
- latest run IDs
- known limitations

The dashboard will not display internal paths, secrets, or raw provider
responses.

In [21]:
metadata_status_code, metadata_payload = (
    request_json(
        f"{FASTAPI_BASE_URL}/metadata"
    )
)

assert metadata_status_code == 200

metadata_schema_df = pd.DataFrame(
    [
        {
            "field": key,
            "python_type": type(
                value
            ).__name__,
            "value": value,
        }
        for key, value
        in metadata_payload.items()
    ]
)

display(metadata_schema_df)

,field,python_type,value
0,project_name,str,Pearls AQI Predictor
1,application_description,str,72-hour PM2.5-based AQI forecast for the Zafar...
2,location,dict,"{'name': 'Zafar Memon DHA', 'latitude': 24.814..."
3,pollution_source,str,OpenAQ
4,original_sensor_provider,str,AirGradient
5,weather_sources,list,"[Open-Meteo Historical Weather API, Open-Meteo..."
6,pollutant,str,PM2.5
7,concentration_unit,str,µg/m³
8,forecast_horizon_hours,int,72
9,internal_timezone,str,UTC


### Inspect operational pipeline status

The System Status page will display the latest Phase 5 and Phase 6 validation
states, row counts, alert counts, and freshness.

This page is intended for technical demonstration and operational visibility.

In [22]:
pipeline_status_code, pipeline_payload = (
    request_json(
        f"{FASTAPI_BASE_URL}/pipeline/status"
    )
)

assert pipeline_status_code == 200

pipeline_schema_df = pd.DataFrame(
    [
        {
            "field": key,
            "python_type": type(
                value
            ).__name__,
            "value": value,
        }
        for key, value
        in pipeline_payload.items()
    ]
)

display(pipeline_schema_df)

,field,python_type,value
0,phase_5_run_id,str,20260728T150416Z_d93945f7
1,phase_5_status,str,PASSED
2,phase_6_run_id,str,20260728T150612Z_aqi_d93945f7
3,phase_6_status,str,AQI_ALERT_PIPELINE_APPROVED
4,generated_at_utc,str,2026-07-28T15:06:12.277712Z
5,artifact_consistency_passed,bool,True
6,forecast_row_count,int,72
7,prediction_count,int,72
8,active_alert_count,int,0
9,alert_episode_count,int,0


### Inspect the FastAPI error contract

The Streamlit API client must convert FastAPI errors into friendly dashboard
states.

Normal users should not see raw stack traces, internal paths, or unformatted
JSON errors.

In [23]:
error_status_code, error_payload = (
    request_json(
        (
            f"{FASTAPI_BASE_URL}/forecast/hourly"
            "?minimum_horizon=30"
            "&maximum_horizon=10"
        )
    )
)

print("Error HTTP status:", error_status_code)

display(
    pd.json_normalize(
        error_payload
    ).T.rename(
        columns={0: "value"}
    )
)

assert error_status_code == 400
assert "error" in error_payload

Error HTTP status: 400


,value
error.code,INVALID_QUERY_PARAMETER
error.message,minimum_horizon cannot be greater than maximum...
error.request_id,ddba4dbf-a486-40fa-9325-10c935f66c14
error.timestamp_utc,2026-07-28T18:26:11.374736+00:00


In [24]:
error_contract = {
    "code": (
        error_payload
        .get("error", {})
        .get("code")
    ),
    "message": (
        error_payload
        .get("error", {})
        .get("message")
    ),
    "details_type": type(
        error_payload
        .get("error", {})
        .get("details")
    ).__name__,
    "request_id_present": bool(
        error_payload
        .get("error", {})
        .get("request_id")
    ),
    "timestamp_present": bool(
        error_payload
        .get("error", {})
        .get("timestamp_utc")
    ),
}

display(
    pd.Series(
        error_contract,
        name="value",
    ).to_frame()
)

,value
code,INVALID_QUERY_PARAMETER
message,minimum_horizon cannot be greater than maximum...
details_type,dict
request_id_present,True
timestamp_present,True


### Confirm supported AQI categories and alert levels

Dashboard filters and colors must use the exact values returned by FastAPI.

The dashboard must not introduce slightly different labels.

In [25]:
observed_aqi_categories = sorted(
    set(
        hourly_forecast_df[
            "indicative_hourly_aqi_category"
        ].dropna()
    )
    | set(
        hourly_forecast_df[
            "rolling_24h_aqi_category"
        ].dropna()
    )
    | set(
        hourly_forecast_df[
            "alert_trigger_category"
        ].dropna()
    )
)

observed_alert_levels = sorted(
    hourly_forecast_df[
        "alert_level"
    ].dropna().unique().tolist()
)

enum_inspection = {
    "observed_aqi_categories": (
        observed_aqi_categories
    ),
    "observed_alert_levels": (
        observed_alert_levels
    ),
}

display(
    pd.Series(
        enum_inspection,
        name="value",
    ).to_frame()
)

,value
observed_aqi_categories,"[Good, Moderate]"
observed_alert_levels,[NORMAL]


In [26]:
SUPPORTED_AQI_CATEGORIES = [
    "Good",
    "Moderate",
    "Unhealthy for Sensitive Groups",
    "Unhealthy",
    "Very Unhealthy",
    "Hazardous",
    "Beyond the AQI",
]

SUPPORTED_ALERT_LEVELS = [
    "NORMAL",
    "ADVISORY",
    "WARNING",
    "SEVERE",
    "EMERGENCY",
]

assert set(
    observed_aqi_categories
).issubset(
    SUPPORTED_AQI_CATEGORIES
)

assert set(
    observed_alert_levels
).issubset(
    SUPPORTED_ALERT_LEVELS
)

### Dashboard information architecture

The first dashboard version will use three pages.

### Forecast

The primary portfolio page containing:

- header and disclaimer
- readiness and freshness
- summary cards
- forecast-range and timezone controls
- PM2.5 forecast chart
- indicative hourly AQI chart
- rolling 24-hour AQI chart
- AQI category timeline
- selected-hour detail
- hourly forecast table
- health guidance

### Alerts

Contains:

- active alert summary
- upcoming alerts
- grouped alert episodes
- hazardous-condition status
- no-alert state
- health actions

### System Status

Contains:

- API liveness and readiness
- forecast freshness
- Phase 5 and Phase 6 statuses
- pipeline run IDs
- forecast row count
- data-source and AQI metadata
- reference-location map
- known limitations

In [27]:
dashboard_page_inventory = [
    {
        "page": "Forecast",
        "route_file": (
            "dashboard/pages/1_Forecast.py"
        ),
        "primary_endpoints": [
            "/forecast",
            "/forecast/summary",
            "/health/ready",
        ],
    },
    {
        "page": "Alerts",
        "route_file": (
            "dashboard/pages/2_Alerts.py"
        ),
        "primary_endpoints": [
            "/alerts",
            "/alerts/active",
            "/forecast/summary",
        ],
    },
    {
        "page": "System Status",
        "route_file": (
            "dashboard/pages/3_System_Status.py"
        ),
        "primary_endpoints": [
            "/health/live",
            "/health/ready",
            "/metadata",
            "/pipeline/status",
        ],
    },
]

dashboard_page_inventory_df = (
    pd.DataFrame(
        dashboard_page_inventory
    )
)

display(dashboard_page_inventory_df)

,page,route_file,primary_endpoints
0,Forecast,dashboard/pages/1_Forecast.py,"[/forecast, /forecast/summary, /health/ready]"
1,Alerts,dashboard/pages/2_Alerts.py,"[/alerts, /alerts/active, /forecast/summary]"
2,System Status,dashboard/pages/3_System_Status.py,"[/health/live, /health/ready, /metadata, /pipe..."


### Dashboard component inventory

Components will be reusable but intentionally limited.

The dashboard will avoid a single oversized page file while also avoiding
unnecessary component fragmentation.

In [28]:
dashboard_component_inventory = [
    {
        "component": "Header",
        "module": (
            "dashboard/components/header.py"
        ),
        "purpose": (
            "Title, location, readiness, freshness, "
            "refresh action, and disclaimer"
        ),
    },
    {
        "component": "Metric cards",
        "module": (
            "dashboard/components/metric_cards.py"
        ),
        "purpose": (
            "PM2.5, AQI, peak, alert, and freshness cards"
        ),
    },
    {
        "component": "Forecast charts",
        "module": (
            "dashboard/components/forecast_charts.py"
        ),
        "purpose": (
            "PM2.5, indicative AQI, rolling AQI, "
            "and category timeline charts"
        ),
    },
    {
        "component": "Alert cards",
        "module": (
            "dashboard/components/alert_cards.py"
        ),
        "purpose": (
            "Active, upcoming, episode, and no-alert states"
        ),
    },
    {
        "component": "Dashboard states",
        "module": (
            "dashboard/components/states.py"
        ),
        "purpose": (
            "Loading, unavailable, stale, empty, "
            "and invalid-response states"
        ),
    },
    {
        "component": "API client",
        "module": (
            "dashboard/services/api_client.py"
        ),
        "purpose": (
            "All FastAPI communication and error parsing"
        ),
    },
    {
        "component": "Formatting utilities",
        "module": (
            "dashboard/utils/formatting.py"
        ),
        "purpose": (
            "PM2.5, AQI, null, duration, and timestamps"
        ),
    },
    {
        "component": "Constants",
        "module": (
            "dashboard/utils/constants.py"
        ),
        "purpose": (
            "Supported categories, alert levels, "
            "and default controls"
        ),
    },
]

dashboard_component_inventory_df = (
    pd.DataFrame(
        dashboard_component_inventory
    )
)

display(
    dashboard_component_inventory_df
)

,component,module,purpose
0,Header,dashboard/components/header.py,"Title, location, readiness, freshness, refresh..."
1,Metric cards,dashboard/components/metric_cards.py,"PM2.5, AQI, peak, alert, and freshness cards"
2,Forecast charts,dashboard/components/forecast_charts.py,"PM2.5, indicative AQI, rolling AQI, and catego..."
3,Alert cards,dashboard/components/alert_cards.py,"Active, upcoming, episode, and no-alert states"
4,Dashboard states,dashboard/components/states.py,"Loading, unavailable, stale, empty, and invali..."
5,API client,dashboard/services/api_client.py,All FastAPI communication and error parsing
6,Formatting utilities,dashboard/utils/formatting.py,"PM2.5, AQI, null, duration, and timestamps"
7,Constants,dashboard/utils/constants.py,"Supported categories, alert levels, and defaul..."


### Endpoint usage contract

Each FastAPI endpoint has a clear dashboard consumer.

The UI will not make scattered raw HTTP requests. All communication will pass
through one API client.

In [29]:
dashboard_endpoint_usage = [
    {
        "endpoint": "/health/live",
        "dashboard_usage": (
            "System Status process health"
        ),
    },
    {
        "endpoint": "/health/ready",
        "dashboard_usage": (
            "Header readiness, freshness, "
            "and stale/not-ready states"
        ),
    },
    {
        "endpoint": "/forecast",
        "dashboard_usage": (
            "Main Forecast page, charts, cards, "
            "table, and selected-hour detail"
        ),
    },
    {
        "endpoint": "/forecast/hourly",
        "dashboard_usage": (
            "Optional server-side filtered forecast requests"
        ),
    },
    {
        "endpoint": "/forecast/summary",
        "dashboard_usage": (
            "Summary cards and alert overview"
        ),
    },
    {
        "endpoint": "/alerts",
        "dashboard_usage": (
            "All grouped alert episodes"
        ),
    },
    {
        "endpoint": "/alerts/active",
        "dashboard_usage": (
            "Current and upcoming alert banners"
        ),
    },
    {
        "endpoint": "/metadata",
        "dashboard_usage": (
            "Location, source, AQI method, "
            "and limitation details"
        ),
    },
    {
        "endpoint": "/pipeline/status",
        "dashboard_usage": (
            "System Status operational information"
        ),
    },
]

dashboard_endpoint_usage_df = pd.DataFrame(
    dashboard_endpoint_usage
)

display(dashboard_endpoint_usage_df)

,endpoint,dashboard_usage
0,/health/live,System Status process health
1,/health/ready,"Header readiness, freshness, and stale/not-rea..."
2,/forecast,"Main Forecast page, charts, cards, table, and ..."
3,/forecast/hourly,Optional server-side filtered forecast requests
4,/forecast/summary,Summary cards and alert overview
5,/alerts,All grouped alert episodes
6,/alerts/active,Current and upcoming alert banners
7,/metadata,"Location, source, AQI method, and limitation d..."
8,/pipeline/status,System Status operational information


### Initial sidebar controls

The Forecast page will initially support:

- forecast range: 12, 24, 48, or 72 hours
- display timezone: Asia/Karachi or UTC
- category filter
- alert-level filter
- alerts-only toggle
- manual refresh

Automatic refresh is deferred until the core dashboard is stable.

This avoids unnecessary API traffic during development.

In [30]:
sidebar_control_inventory = [
    {
        "control": "Forecast range",
        "type": "Selectbox",
        "values": [12, 24, 48, 72],
        "default": 24,
    },
    {
        "control": "Timezone",
        "type": "Selectbox",
        "values": [
            "Asia/Karachi",
            "UTC",
        ],
        "default": "Asia/Karachi",
    },
    {
        "control": "AQI category",
        "type": "Multiselect",
        "values": SUPPORTED_AQI_CATEGORIES,
        "default": [],
    },
    {
        "control": "Alert level",
        "type": "Multiselect",
        "values": SUPPORTED_ALERT_LEVELS,
        "default": [],
    },
    {
        "control": "Alerts only",
        "type": "Toggle",
        "values": [False, True],
        "default": False,
    },
    {
        "control": "Manual refresh",
        "type": "Button",
        "values": None,
        "default": None,
    },
]

sidebar_control_inventory_df = (
    pd.DataFrame(
        sidebar_control_inventory
    )
)

display(sidebar_control_inventory_df)

,control,type,values,default
0,Forecast range,Selectbox,"[12, 24, 48, 72]",24
1,Timezone,Selectbox,"[Asia/Karachi, UTC]",Asia/Karachi
2,AQI category,Multiselect,"[Good, Moderate, Unhealthy for Sensitive Group...",[]
3,Alert level,Multiselect,"[NORMAL, ADVISORY, WARNING, SEVERE, EMERGENCY]",[]
4,Alerts only,Toggle,"[False, True]",False
5,Manual refresh,Button,None,None


### Save the initial dashboard contracts

These reports document the final dashboard architecture before implementation.

They will be updated during Phase 8F with final validation results.

In [31]:
endpoint_usage_report_path = (
    REPORTS_DIR
    / "dashboard_endpoint_usage.json"
)

component_inventory_report_path = (
    REPORTS_DIR
    / "dashboard_component_inventory.json"
)

with endpoint_usage_report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        dashboard_endpoint_usage,
        file,
        indent=2,
    )

with component_inventory_report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        dashboard_component_inventory,
        file,
        indent=2,
    )

print(
    "Endpoint usage report:",
    endpoint_usage_report_path,
)

print(
    "Component inventory report:",
    component_inventory_report_path,
)

Endpoint usage report: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/dashboard_endpoint_usage.json
Component inventory report: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/dashboard_component_inventory.json


## **8B.** Dashboard configuration, API client, and utilities

This subphase creates the non-visual foundation of the Streamlit dashboard.

It includes:

- environment-based dashboard configuration
- one reusable FastAPI HTTP client
- safe retries for GET requests
- timeout and connection-error handling
- structured FastAPI error parsing
- short-lived Streamlit caching
- manual cache clearing
- timezone-aware timestamp conversion
- PM2.5, AQI, freshness, duration, and null formatting
- validated conversion of forecast records into a dashboard DataFrame
- reusable dashboard-side display filtering

No Streamlit page, chart, card, or alert component is created yet.

In [32]:
from dashboard.config import (
    get_dashboard_settings,
)
from dashboard.services.api_client import (
    FastAPIClient,
)

dashboard_settings = (
    get_dashboard_settings()
)

dashboard_api_client = FastAPIClient(
    dashboard_settings
)

settings_summary = {
    "fastapi_base_url": (
        dashboard_settings.fastapi_base_url
    ),
    "request_timeout_seconds": (
        dashboard_settings
        .dashboard_request_timeout_seconds
    ),
    "cache_ttl_seconds": (
        dashboard_settings
        .dashboard_cache_ttl_seconds
    ),
    "dashboard_title": (
        dashboard_settings.dashboard_title
    ),
    "default_timezone": (
        dashboard_settings
        .dashboard_default_timezone
    ),
    "environment": (
        dashboard_settings
        .dashboard_environment
    ),
}

display(
    pd.Series(
        settings_summary,
        name="value",
    ).to_frame()
)

,value
fastapi_base_url,http://localhost:8000/api/v1
request_timeout_seconds,10.0
cache_ttl_seconds,60
dashboard_title,Pearls AQI Predictor
default_timezone,Asia/Karachi
environment,development


In [33]:
dashboard_liveness = (
    dashboard_api_client.get_liveness()
)

dashboard_readiness = (
    dashboard_api_client.get_readiness()
)

dashboard_forecast = (
    dashboard_api_client.get_forecast()
)

dashboard_summary = (
    dashboard_api_client.get_summary()
)

dashboard_alerts = (
    dashboard_api_client.get_alerts()
)

dashboard_metadata = (
    dashboard_api_client.get_metadata()
)

dashboard_pipeline = (
    dashboard_api_client
    .get_pipeline_status()
)

api_client_validation_summary = {
    "liveness": (
        dashboard_liveness.get("status")
    ),
    "readiness": (
        dashboard_readiness.get("status")
    ),
    "forecast_rows": len(
        dashboard_forecast.get(
            "hourly_forecast",
            [],
        )
    ),
    "summary_run_id": (
        dashboard_summary.get(
            "phase_6_run_id"
        )
    ),
    "alert_episodes": (
        dashboard_alerts.get(
            "episode_count"
        )
    ),
    "metadata_location": (
        dashboard_metadata
        .get("location", {})
        .get("name")
    ),
    "pipeline_rows": (
        dashboard_pipeline.get(
            "forecast_row_count"
        )
    ),
}

display(
    pd.Series(
        api_client_validation_summary,
        name="value",
    ).to_frame()
)

,value
liveness,ALIVE
readiness,READY_WITH_LIMITATIONS
forecast_rows,72
summary_run_id,20260728T150612Z_aqi_d93945f7
alert_episodes,0
metadata_location,Zafar Memon DHA
pipeline_rows,72


In [34]:
from dashboard.utils.data import (
    add_display_timezone,
    filter_hourly_forecast,
    prepare_hourly_forecast,
)

prepared_forecast_df = (
    prepare_hourly_forecast(
        dashboard_forecast
    )
)

prepared_forecast_df = (
    add_display_timezone(
        prepared_forecast_df,
        timezone_name="Asia/Karachi",
    )
)

filtered_24h_df = (
    filter_hourly_forecast(
        prepared_forecast_df,
        maximum_horizon=24,
    )
)

data_preparation_summary = {
    "prepared_rows": len(
        prepared_forecast_df
    ),
    "filtered_24h_rows": len(
        filtered_24h_df
    ),
    "first_horizon": int(
        prepared_forecast_df[
            "forecast_horizon_hours"
        ].iloc[0]
    ),
    "last_horizon": int(
        prepared_forecast_df[
            "forecast_horizon_hours"
        ].iloc[-1]
    ),
    "first_utc_time": (
        prepared_forecast_df[
            "target_time_utc"
        ].iloc[0]
    ),
    "first_karachi_time": (
        prepared_forecast_df[
            "display_time"
        ].iloc[0]
    ),
}

display(
    pd.Series(
        data_preparation_summary,
        name="value",
    ).to_frame()
)

,value
prepared_rows,72
filtered_24h_rows,24
first_horizon,1
last_horizon,72
first_utc_time,2026-07-28 14:00:00+00:00
first_karachi_time,2026-07-28 19:00:00+05:00


In [35]:
from dashboard.utils.formatting import (
    format_aqi,
    format_freshness,
    format_pm25,
    format_timestamp,
)

formatting_examples = {
    "pm25": format_pm25(14.34),
    "aqi": format_aqi(61),
    "missing_rolling_aqi": format_aqi(
        None
    ),
    "karachi_timestamp": format_timestamp(
        dashboard_forecast[
            "forecast_start_utc"
        ],
        timezone_name="Asia/Karachi",
    ),
    "utc_timestamp": format_timestamp(
        dashboard_forecast[
            "forecast_start_utc"
        ],
        timezone_name="UTC",
    ),
    "freshness": format_freshness(
        status=dashboard_forecast[
            "freshness"
        ]["status"],
        age_hours=dashboard_forecast[
            "freshness"
        ]["age_hours"],
    ),
}

display(
    pd.Series(
        formatting_examples,
        name="value",
    ).to_frame()
)

,value
pm25,14.3 µg/m³
aqi,61
missing_rolling_aqi,Not available
karachi_timestamp,"28 Jul 2026, 07:00 PM PKT"
utc_timestamp,"28 Jul 2026, 02:00 PM UTC"
freshness,FRESH · 3.3 hours old


## **8C.**  Professional Forecast dashboard

The primary Streamlit page is now implemented.

It consumes the FastAPI forecast and readiness endpoints and provides:

- readiness and freshness status
- a visible forecast disclaimer
- forecast range and timezone controls
- category and alert-level filters
- manual cache refresh
- summary metric cards
- interactive PM2.5 and AQI charts
- rolling 24-hour AQI availability handling
- a category timeline
- forecast-hour detail interaction
- health guidance
- a formatted hourly table
- safe empty, stale, and API-error states

The dashboard does not read pipeline artifacts or perform AQI calculations.

In [36]:
from dashboard.components.forecast_charts import (
    build_category_timeline,
    build_indicative_aqi_chart,
    build_pm25_chart,
    build_rolling_aqi_chart,
)
from dashboard.pages.forecast import (
    render_forecast_page,
)

print(
    "Phase 8C modules imported successfully."
)

Phase 8C modules imported successfully.


In [37]:
phase_8c_chart_df = (
    prepared_forecast_df
    .head(24)
    .copy()
)

pm25_chart = build_pm25_chart(
    phase_8c_chart_df
)

indicative_chart = (
    build_indicative_aqi_chart(
        phase_8c_chart_df
    )
)

rolling_chart = build_rolling_aqi_chart(
    phase_8c_chart_df
)

timeline_chart = build_category_timeline(
    phase_8c_chart_df
)

chart_validation_summary = {
    "pm25_trace_count": len(
        pm25_chart.data
    ),
    "indicative_trace_count": len(
        indicative_chart.data
    ),
    "rolling_chart_available": (
        rolling_chart is not None
    ),
    "timeline_trace_count": len(
        timeline_chart.data
    ),
}

display(
    pd.Series(
        chart_validation_summary,
        name="value",
    ).to_frame()
)

assert chart_validation_summary[
    "pm25_trace_count"
] >= 1

assert chart_validation_summary[
    "indicative_trace_count"
] >= 1

assert chart_validation_summary[
    "timeline_trace_count"
] >= 1

,value
pm25_trace_count,1
indicative_trace_count,1
rolling_chart_available,True
timeline_trace_count,2


## **8D.** Alerts and System Status pages

The Streamlit dashboard now includes two additional pages.

The Alerts page provides:

- alert episode totals
- currently active alerts
- upcoming alerts
- hazardous episode counts
- grouped episode details
- health recommendations
- a clear normal-condition state

The System Status page provides:

- API availability
- readiness and freshness
- Phase 5 and Phase 6 statuses
- run identifiers
- forecast and alert counts
- public project metadata
- reference-location mapping
- optional technical notes

Technical limitations are kept away from the primary Forecast page and are shown
only in the operational System Status view.

In [38]:
from dashboard.pages import (
    render_alerts_page,
    render_forecast_page,
    render_system_status_page,
)

from dashboard.services.api_client import (
    FastAPIClient,
)

phase_8d_client = FastAPIClient()

phase_8d_alerts = (
    phase_8d_client.get_alerts()
)

phase_8d_active_alerts = (
    phase_8d_client.get_active_alerts()
)

phase_8d_metadata = (
    phase_8d_client.get_metadata()
)

phase_8d_pipeline = (
    phase_8d_client.get_pipeline_status()
)

phase_8d_validation_summary = {
    "alert_episode_count": (
        phase_8d_alerts.get(
            "episode_count"
        )
    ),
    "current_alert_count": (
        phase_8d_active_alerts.get(
            "current_count"
        )
    ),
    "upcoming_alert_count": (
        phase_8d_active_alerts.get(
            "upcoming_count"
        )
    ),
    "location": (
        phase_8d_metadata
        .get("location", {})
        .get("name")
    ),
    "forecast_rows": (
        phase_8d_pipeline.get(
            "forecast_row_count"
        )
    ),
    "phase_6_status": (
        phase_8d_pipeline.get(
            "phase_6_status"
        )
    ),
}

display(
    pd.Series(
        phase_8d_validation_summary,
        name="value",
    ).to_frame()
)

assert callable(
    render_forecast_page
)

assert callable(
    render_alerts_page
)

assert callable(
    render_system_status_page
)

assert isinstance(
    phase_8d_alerts.get(
        "episodes"
    ),
    list,
)

assert (
    phase_8d_pipeline.get(
        "forecast_row_count"
    )
    == 72
)

assert (
    phase_8d_metadata
    .get("location", {})
    .get("latitude")
    is not None
)

print(
    "Phase 8D Alerts and System Status "
    "dashboard validation passed."
)

,value
alert_episode_count,0
current_alert_count,0
upcoming_alert_count,0
location,Zafar Memon DHA
forecast_rows,72
phase_6_status,AQI_ALERT_PIPELINE_APPROVED


Phase 8D Alerts and System Status dashboard validation passed.


## **8E.** Dashboard tests

The dashboard's essential non-visual behavior is now covered by automated tests:

- successful FastAPI responses
- structured API failures
- request timeouts
- PM2.5, AQI, freshness, and timestamp formatting
- UTC-to-Karachi conversion
- forecast payload preparation
- 24-hour horizon filtering
- category and alerts-only filtering
- invalid forecast payload handling

## **8F.** Docker Compose and final dashboard validation

The completed dashboard and FastAPI service can now run together through Docker
Compose.

Confirmed architecture:

- FastAPI mounts and validates the latest Phase 6 artifacts
- Streamlit communicates with FastAPI over the Compose network
- Streamlit does not read pipeline artifacts directly
- API and dashboard containers expose independent health checks
- configuration is supplied through environment variables
- dashboard pages remain available through one Streamlit application

In [39]:
from pathlib import Path
import json

import requests


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


final_dashboard_checks = {
    "dashboard_dockerfile_exists": (
        PROJECT_ROOT
        / "dashboard"
        / "Dockerfile"
    ).exists(),
    "compose_file_exists": (
        PROJECT_ROOT
        / "compose.yaml"
    ).exists(),
    "streamlit_config_exists": (
        PROJECT_ROOT
        / ".streamlit"
        / "config.toml"
    ).exists(),
}


api_live_response = requests.get(
    "http://localhost:8000/api/v1/health/live",
    timeout=10,
)

api_ready_response = requests.get(
    "http://localhost:8000/api/v1/health/ready",
    timeout=10,
)

dashboard_health_response = requests.get(
    "http://localhost:8501/_stcore/health",
    timeout=10,
)


final_dashboard_checks.update(
    {
        "api_liveness_status": (
            api_live_response.status_code
        ),
        "api_readiness_status": (
            api_ready_response.status_code
        ),
        "dashboard_health_status": (
            dashboard_health_response.status_code
        ),
    }
)


display(
    pd.Series(
        final_dashboard_checks,
        name="value",
    ).to_frame()
)

,value
dashboard_dockerfile_exists,True
compose_file_exists,True
streamlit_config_exists,True
api_liveness_status,200
api_readiness_status,200
dashboard_health_status,200


In [40]:
REPORTS_DIR = PROJECT_ROOT / "reports"

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

phase_8_approved = all(
    [
        final_dashboard_checks[
            "dashboard_dockerfile_exists"
        ],
        final_dashboard_checks[
            "compose_file_exists"
        ],
        final_dashboard_checks[
            "streamlit_config_exists"
        ],
        final_dashboard_checks[
            "api_liveness_status"
        ] == 200,
        final_dashboard_checks[
            "api_readiness_status"
        ] in {200, 503},
        final_dashboard_checks[
            "dashboard_health_status"
        ] == 200,
    ]
)

phase_8_validation_report = {
    "dashboard_name": "Pearls AQI Predictor",
    "dashboard_framework": "Streamlit",
    "dashboard_port": 8501,
    "api_port": 8000,
    "dashboard_pages": [
        "Forecast",
        "Alerts",
        "System Status",
    ],
    "fastapi_is_single_data_source": True,
    "dashboard_reads_artifacts_directly": False,
    "forecast_page_completed": True,
    "alerts_page_completed": True,
    "system_status_page_completed": True,
    "dashboard_tests_completed": True,
    "docker_support_completed": True,
    "compose_support_completed": True,
    "dashboard_health_passed": (
        dashboard_health_response.status_code
        == 200
    ),
    "api_health_passed": (
        api_live_response.status_code
        == 200
    ),
    "final_phase_8_status": (
        "STREAMLIT_DASHBOARD_APPROVED"
        if phase_8_approved
        else "STREAMLIT_DASHBOARD_NOT_READY"
    ),
}

phase_8_report_path = (
    REPORTS_DIR
    / "phase_8_dashboard_validation_report.json"
)

with phase_8_report_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        phase_8_validation_report,
        file,
        indent=2,
    )

display(
    pd.Series(
        phase_8_validation_report,
        name="value",
    ).to_frame()
)

print(
    "Phase 8 report saved:",
    phase_8_report_path,
)

,value
dashboard_name,Pearls AQI Predictor
dashboard_framework,Streamlit
dashboard_port,8501
api_port,8000
dashboard_pages,"[Forecast, Alerts, System Status]"
fastapi_is_single_data_source,True
dashboard_reads_artifacts_directly,False
forecast_page_completed,True
alerts_page_completed,True
system_status_page_completed,True


Phase 8 report saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/reports/phase_8_dashboard_validation_report.json
